# results tables and figures
generates all tables and figure 1 from the inference pipeline outputs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from scipy.stats import binomtest, chi2_contingency
from statsmodels.stats.contingency_tables import mcnemar
import warnings; warnings.filterwarnings('ignore')
print('ready')

In [ ]:
# paths to the results files
INFERENCE= '/kaggle/input/datasets/zainabrizvi5/thesis-inference-results/bias_results_v2_full.csv'
RAG= '/kaggle/input/datasets/zainabrizvi5/thesis-inference-results/rag_bias_results_v2.csv'
OUT = '/kaggle/working/'
df= pd.read_csv(INFERENCE)
rag_df= pd.read_csv(RAG)
print(f'loaded: {df.shape}, {rag_df.shape}')

## table 1: classifier label distributions

In [ ]:
# what each classifier labeled each llm's justification text as
rows= []
for country in ['US', 'UK']:
    subset= df[df['country'] ==country]
    for llm in ['gpt', 'claude','llama']:
        llm_df= subset[subset['llm'] == llm]
        row= {'LLM': llm.upper(), 'Country': country}
        for model in ['politbert', 'politics', 't5']:
            col= f'{model}_label'
            counts= llm_df[col].value_counts()
            pcts= llm_df[col].value_counts(normalize=True).mul(100).round(1)
            row[model.upper()]= ' | '.join([f'{l}: {counts[l]} ({pcts[l]}%)' for l in counts.index])
        rows.append(row)
table1= pd.DataFrame(rows)
print(table1.to_string(index=False))
table1.to_csv(OUT + 'table1_classifier_label_distributions.csv',index=False)

## table 2: classifier performance

In [ ]:
# test set results from training runs
perf= {
    'PolitBERT': {'US': (64.22, 64.19), 'UK': (63.46, 56.06)},
    'POLITICS':  {'US': (66.12, 66.12),'UK': (65.20, 57.36)},
    'T5': {'US': (60.37, 60.29), 'UK': (51.51, 48.40)},
}
rows= []
for model, vals in perf.items():
    for country, (acc, f1) in vals.items():
        classes = 'left / right' if country == 'US' else 'left / center / right'
        rows.append({'model': model, 'dataset': country, 'classes': classes, 'accuracy': acc, 'macro f1': f1})
table2 = pd.DataFrame(rows)
print(table2.to_string(index=False))
table2.to_csv(OUT + 'table2_classifier_performance.csv',index=False)

## figure 1: macro f1 by classifier

In [ ]:
models= ['PolitBERT', 'POLITICS (best)', 'T5 (weakest)']
us_f1 = [64.19, 66.12, 60.29]
uk_f1= [56.06, 57.36, 48.40]
x, width= range(len(models)), 0.35
fig, ax= plt.subplots(figsize=(6.4, 3.4))
bars1= ax.bar([i -width/2 for i in x], us_f1, width, label='US (binary)', color='#85B7EB', zorder=3)
bars2= ax.bar([i + width/2 for i in x], uk_f1, width, label='UK (3-class)', color='#185FA5', zorder=3)
ax.set_ylim(40, 70)
ax.set_xticks(list(x))
ax.set_xticklabels(models, fontsize=11,fontweight='medium',color='#1B4F72')
ax.tick_params(axis='y', labelsize=9,colors='#52514e')
ax.yaxis.set_major_formatter(lambda v,pos: f'{int(v)}%')
for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
ax.grid(axis='y', color='#e1e0d9',linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for bars in [bars1, bars2]:
    for bar in bars:
        h= bar.get_height()
        ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8.5, color='#1B4F72')
ax.legend(loc='upper left', frameon=False, fontsize=9, labelcolor='#52514e',bbox_to_anchor=(0, 1.08))
ax.set_title('Macro F1 Score by Classifier', fontsize=13, fontweight='bold', color='#1B4F72', pad=20)
plt.tight_layout(pad=0.6)
plt.savefig(OUT + 'figure1_classifier_performance.pdf', bbox_inches='tight', pad_inches=0.15)
plt.savefig(OUT + 'figure1_classifier_performance.png', bbox_inches='tight', dpi=150)
plt.show()

## table 3: llm neutrality rates

In [ ]:
llm_names = {'gpt': 'GPT-4o-mini', 'claude': 'Claude Haiku 4.5', 'llama': 'Llama 3.3-70B'}
rows = []
for llm_key, llm_name in llm_names.items():
    for country in ['US', 'UK', 'Overall']:
        sub= (df[df['llm']==llm_key] if country=='Overall'
              else df[(df['llm']==llm_key) & (df['country']==country)])
        n= len(sub)
        cc= sub['choice'].value_counts()
        a, b, neu= cc.get('A',0), cc.get('B',0), cc.get('neutral',0)
        rows.append({'llm': llm_name, 'country': country,
                     'choice a': f'{a} ({a/n*100:.0f}%)',
                     'choice b': f'{b} ({b/n*100:.0f}%)',
                     'neutral': f'{neu} ({neu/n*100:.0f}%)',
                     'total': n})
table3 = pd.DataFrame(rows)
print(table3.to_string(index=False))
# chi-square test : does llm identity predict choice?
ct = pd.crosstab(df['llm'], df['choice'])
chi2, p, dof, exp= chi2_contingency(ct)
print(f'\nchi-square: chi2={chi2:.2f}, df={dof}, p={p:.8f}, min expected={exp.min():.2f}')
table3.to_csv(OUT + 'table3_neutrality_rates.csv', index=False)

## table 4: stated neutrality vs classified lean

In [ ]:
# claude said neutral: what did politbert actually classify the justification as
claude_df = df[df['llm']=='claude']
rows= []
for country in ['US', 'UK']:
    neu= claude_df[(claude_df['country']==country) & (claude_df['choice']=='neutral')]
    n= len(neu)
    counts= neu['politbert_label'].value_counts()
    pcts= neu['politbert_label'].value_counts(normalize=True).mul(100).round(1)
    row= {'country': country, 'neutral (n)': n}
    for label in ['left','center','right']:
        if label=='center' and country=='US':
            row[f'classified {label}']= 'n/a'
        else:
            c= counts.get(label, 0)
            row[f'classified {label}']= f'{c} ({pcts.get(label,0.0):.0f}%)'
    rows.append(row)
all_neu= claude_df[claude_df['choice']=='neutral']
n_all = len(all_neu)
c_all= all_neu['politbert_label'].value_counts()
p_all= all_neu['politbert_label'].value_counts(normalize=True).mul(100).round(1)
row= {'country': 'overall', 'neutral (n)': n_all}
for label in ['left','center','right']:
    c= c_all.get(label,0)
    row[f'classified {label}']= f'{c} ({p_all.get(label,0.0):.0f}%)' + ('*' if label=='center' else '')
rows.append(row)

table4= pd.DataFrame(rows)
print(table4.to_string(index=False))

# binomial test - is 15/18 left significantly above chance?
us_neu = claude_df[(claude_df['country']=='US') & (claude_df['choice']=='neutral')]
left_n= (us_neu['politbert_label']=='left').sum()
r= binomtest(left_n, n=len(us_neu), p=0.5, alternative='two-sided')
print(f'\nbinomial test (us neutral -> left): {left_n}/{len(us_neu)}, p={r.pvalue:.4f}')
table4.to_csv(OUT + 'table4_stated_neutrality_vs_classified.csv', index=False)

## table 5: rag vs standard

In [ ]:
rows= []
for model in ['politbert','politics','t5']:
    chg= rag_df[f'{model}_changed'].sum()
    n= len(rag_df)
    rows.append({'classifier': model.upper(), 'labels changed': f'{chg}/{n}', '%': f'{chg/n*100:.0f}%'})
table5 = pd.DataFrame(rows)
print(table5.to_string(index=False))
# mcnemar test on center class collapse in uk
uk_rag= rag_df[rag_df['country']=='UK']
for model in ['politbert','t5']:
    std= (uk_rag[f'{model}_std']=='center').astype(int)
    rag= (uk_rag[f'{model}_rag']=='center').astype(int)
    a= ((std==1)&(rag==1)).sum()
    b= ((std==1)&(rag==0)).sum()
    c= ((std==0)&(rag==1)).sum()
    d= ((std==0)&(rag==0)).sum()
    r= mcnemar([[a,b],[c,d]], exact=(b+c)<25, correction=True)
    print(f'mcnemar ({model}, uk center): p={r.pvalue:.4f}')
table5.to_csv(OUT + 'table5_rag_vs_standard.csv', index=False)

## table 6: classifier agreement

In [ ]:
pairs= [
    ('politbert','politics','PolitBERT <-> POLITICS'),
    ('politbert','t5','PolitBERT <-> T5'),
    ('politics','t5','POLITICS <-> T5')
]
for country, chance in [('US', 0.5), ('UK', 1/3)]:
    print(f'{country}:')
    sub= df[df['country']==country]
    rows= []
    for m1, m2, label in pairs:
        agree_n= (sub[f'{m1}_label']==sub[f'{m2}_label']).sum()
        n= len(sub)
        kappa= cohen_kappa_score(sub[f'{m1}_label'], sub[f'{m2}_label'])
        r= binomtest(agree_n, n=n, p=chance, alternative='greater')
        rows.append({'pair': label, 'agreement': f'{agree_n/n:.1%}',
                     'kappa': f'{kappa:.3f}', 'p': f'{r.pvalue:.6f}',
                     'sig': 'yes' if r.pvalue < 0.05 else 'no'})
    t= pd.DataFrame(rows)
    print(t.to_string(index=False))
    t.to_csv(OUT + f'table6_agreement_{country.lower()}.csv', index=False)
    print()

## complete


In [ ]:
import os
files= ['table1_classifier_label_distributions.csv',
        'table2_classifier_performance.csv',
        'table3_neutrality_rates.csv',
        'table4_stated_neutrality_vs_classified.csv',
        'table5_rag_vs_standard.csv',
        'table6_agreement_us.csv',
        'table6_agreement_uk.csv',
        'figure1_classifier_performance.pdf']

print('outputs:')
for f in files:
    exists= os.path.exists(OUT + f)
    print(f' {f} - {"ok" if exists else "missing"}')